# nb19 - Save model predictions to CSV

after inference, save per-cluster predictions to CSV for **every architecture x both datasets** with columns true energy, predicted energy (+ bias), ECAL region (from the seed cell), and **ET**. These feed the comparison plots (nb20).

**Phase 0** de-risks (ET fields, region tiers, timing, schema). **Phase 1** trains all 8 models on both datasets (5 seeds) with a single unified pipeline and writes `reports/predictions/{dataset}__{model}.csv`.

In [1]:
import os, sys, pathlib, copy, collections
import numpy as np, pandas as pd, uproot, awkward as ak
import torch, torch.nn as nn
from sklearn.ensemble import HistGradientBoostingRegressor
REPO = pathlib.Path(os.environ['REPO_DIR']) if os.environ.get('REPO_DIR') else (
    pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd())
sys.path.insert(0, str(REPO / 'scripts'))
from run_experiments import build, select_knn, split, resolution, derive_geom, PITCH, EPS, CELL_KEYS
DATASETS = {
    'clean':   sorted((REPO / 'data' / 'full').glob('matched_*.root')),
    'minbias': sorted((REPO / 'data' / 'minimum_bias').glob('matched_*.root')),
}
OUT = REPO / 'reports' / 'predictions'; OUT.mkdir(parents=True, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODE = os.environ.get('NB19_MODE', 'full')
print('device', DEVICE, '| mode', MODE, '| PITCH', PITCH)
print({k: len(v) for k, v in DATASETS.items()})

device cuda | mode full | PITCH [ 15.  30.  40.  60. 120.]
{'clean': 100, 'minbias': 94}


## T1 - ET fields (Felipe's formula)
`ET = sig_flux_eTot * pT / p`, `pT=hypot(px,py)`, `p=hypot(px,py,pz)` - no z_calo/vertex ambiguity.

In [2]:
MOM = ['sig_flux_px','sig_flux_py','sig_flux_pz','sig_flux_eTot']
for name, files in DATASETS.items():
    with uproot.open(files[0]) as f:
        keys = set(f['clusters_matched'].keys())
    print(name, '->', [k for k in MOM if k in keys], '| missing:', [k for k in MOM if k not in keys])

clean -> ['sig_flux_px', 'sig_flux_py', 'sig_flux_pz', 'sig_flux_eTot'] | missing: []
minbias -> ['sig_flux_px', 'sig_flux_py', 'sig_flux_pz', 'sig_flux_eTot'] | missing: []


In [3]:
def read_momentum(files):
    a = uproot.concatenate([f'{f}:clusters_matched' for f in files], MOM, library='np')
    px, py, pz, e = a['sig_flux_px'], a['sig_flux_py'], a['sig_flux_pz'], a['sig_flux_eTot']
    return e, e * np.hypot(px, py) / np.maximum(np.sqrt(px**2 + py**2 + pz**2), EPS)
for name in ['minbias', 'clean']:
    e, et = read_momentum(DATASETS[name][:3])
    print(f'{name:8s} n={len(et):6d}  eTot med={np.nanmedian(e):8.2f}  ET med={np.nanmedian(et):8.2f}  '
          f'ET<=eTot frac={np.mean((et <= e + 1e-6) & (et >= 0)):.4f}  n_nan={np.isnan(et).sum()}')

minbias  n=  2973  eTot med=   27.70  ET med=    2.80  ET<=eTot frac=1.0000  n_nan=0
clean    n=  5445  eTot med=    3.97  ET med=    0.28  ET<=eTot frac=1.0000  n_nan=0


## T2 - Region label from the seed cell
`region = mod[seed]`, pitch = nearest-neighbour spacing in the seed's module. Five granularity tiers (15/30/40/60/120 mm) are all populated, so we label by pitch tier. **Open for Felipe:** five tiers or three physical zones?

In [4]:
d_reg = build(DATASETS['minbias'][:5], None, 100, selector=lambda cc: select_knn(cc, 25))
reg = np.asarray(d_reg['region'])
for i in sorted(set(reg.tolist())):
    m = reg == i
    print(f'  idx {i} (pitch {PITCH[i]:>3} mm): {m.sum():6d}  ({100*m.mean():5.1f}%)')
REGION_NAME = {i: f'{int(PITCH[i])}mm' for i in range(len(PITCH))}
print('REGION_NAME =', REGION_NAME)

  idx 0 (pitch 15.0 mm):    846  ( 17.3%)
  idx 1 (pitch 30.0 mm):   1035  ( 21.2%)
  idx 2 (pitch 40.0 mm):   1233  ( 25.3%)
  idx 3 (pitch 60.0 mm):   1532  ( 31.4%)
  idx 4 (pitch 120.0 mm):    237  (  4.9%)
REGION_NAME = {0: '15mm', 1: '30mm', 2: '40mm', 3: '60mm', 4: '120mm'}


## T3 - Timing availability (both datasets)
Spacetime models need per-cell timing. Check valid (non-sentinel) fraction on both datasets.

In [5]:
TSENT = 1e6
for name, files in DATASETS.items():
    a = uproot.open(files[0])['clusters_matched'].arrays(['cell_times_front'], library='ak')
    t = ak.to_numpy(ak.flatten(a['cell_times_front']))
    print(f'{name:8s} cell_times_front valid(|t|<{TSENT:.0e}) frac = {np.mean(np.abs(t) < TSENT):.4f}')
CLEAN_HAS_TIMING = True

clean    cell_times_front valid(|t|<1e+06) frac = 0.3303
minbias  cell_times_front valid(|t|<1e+06) frac = 0.3118


# Phase 1 - unified pipeline, train + save

One `build_st` per dataset yields both space tokens (`tok12`, 12-dim) and spacetime tokens (`tok15`, 15-dim with dtf/dtb/hasv) plus pairwise features `R` (5-dim incl. dt, hasv). All 6 transformer variants + 2 baselines train off this single build, so the comparison is on identical footing.

**ET alignment:** `align_et` reproduces `build_st`'s exact keep/skip (`vz<vertex`, empty-after-selector) so ET lines up 1:1 with the built clusters; every prep asserts `len(et)==N`.

In [6]:
def align_et(files, vertex_max, selector):
    ets = []
    for path in files:
        with uproot.open(path) as f:
            a = f['clusters_matched'].arrays(
                CELL_KEYS + ['sig_flux_prod_vertex_z','sig_flux_px','sig_flux_py','sig_flux_pz','sig_flux_eTot'],
                library='ak')
        vz = ak.to_numpy(a['sig_flux_prod_vertex_z']).astype(float)
        px = ak.to_numpy(a['sig_flux_px']).astype(float); py = ak.to_numpy(a['sig_flux_py']).astype(float)
        pz = ak.to_numpy(a['sig_flux_pz']).astype(float); e = ak.to_numpy(a['sig_flux_eTot']).astype(float)
        et = e * np.hypot(px, py) / np.maximum(np.sqrt(px**2 + py**2 + pz**2), EPS)
        for i in np.flatnonzero(vz < vertex_max):
            ci = {k: np.asarray(ak.to_numpy(a[k][i])).astype(float) for k in CELL_KEYS}
            if len(ci['energy'][selector(ci)]) == 0:
                continue
            ets.append(float(et[i]))
    return np.array(ets)

In [7]:
TKEYS = CELL_KEYS + ['cell_times_front', 'cell_times_back']
AUX = ['sig_flux_prod_vertex_z', 'sig_flux_eTot', 'total_energy', 'x_cluster', 'y_cluster']
def build_st(files, selector, vertex_max=100.0):
    O = {k: [] for k in ['tok12','tok15','R','agg','total_energy','y','Etrue','region']}
    for path in files:
        with uproot.open(path) as f:
            a = f['clusters_matched'].arrays(TKEYS + AUX, library='ak')
        vz = ak.to_numpy(a['sig_flux_prod_vertex_z']).astype(float)
        for i in np.flatnonzero(vz < vertex_max):
            cc = {k: np.asarray(ak.to_numpy(a[k][i])).astype(float) for k in TKEYS}
            sel = selector(cc); cw = {k: v[sel] for k, v in cc.items()}
            e = cw['energy']
            if len(e) == 0:
                continue
            pitch, mod, rx, ry, rdr, seed = derive_geom(cw)
            fr = cw['cell_energies_front']; bk = cw['cell_energies_back']
            tf = cw['cell_times_front']; tb = cw['cell_times_back']
            vf = np.abs(tf) < 1e6; vb = np.abs(tb) < 1e6
            t0f = tf[seed] if vf[seed] else (np.median(tf[vf]) if vf.any() else 0.0)
            t0b = tb[seed] if vb[seed] else (np.median(tb[vb]) if vb.any() else 0.0)
            dtf = np.where(vf, tf - t0f, 0.0); dtb = np.where(vb, tb - t0b, 0.0)
            hasv = (vf | vb).astype(float)
            cont7 = np.stack([np.log1p(np.clip(e,0,None)), np.log1p(np.clip(fr,0,None)),
                              np.log1p(np.clip(bk,0,None)), rx/pitch, ry/pitch, rdr/pitch, np.log(pitch)], 1)
            oh = np.zeros((len(e), len(PITCH))); oh[np.arange(len(e)), mod] = 1.0
            O['tok12'].append(np.concatenate([cont7, oh], 1).astype(np.float32))
            O['tok15'].append(np.concatenate([cont7, dtf[:,None], dtb[:,None], hasv[:,None], oh], 1).astype(np.float32))
            O['R'].append(np.stack([rx/pitch, ry/pitch, np.log1p(np.clip(e,0,None)), dtf, hasv], 1).astype(np.float32))
            sumE = float(e.sum()); seedE = float(e[seed]); region = int(mod[seed])
            lat = float(np.sqrt((e * rdr**2).sum() / (sumE + EPS))); fb = float(fr.sum() / (bk.sum() + EPS))
            O['agg'].append([np.log1p(sumE), fb, len(e), np.log1p(seedE), lat, region])
            O['total_energy'].append(float(a['total_energy'][i]))
            et = float(a['sig_flux_eTot'][i]); O['y'].append(np.log(max(et, 1e-3)))
            O['Etrue'].append(et); O['region'].append(region)
    for k in ['agg','total_energy','y','Etrue','region']:
        O[k] = np.array(O[k])
    return O

In [8]:
def save_predictions(pred_energy, pred_bias, data, et, model, dataset, seed, test_idx, write=True):
    ti = np.asarray(test_idx); reg = np.asarray(data['region'])[ti]
    df = pd.DataFrame({
        'model': model, 'dataset': dataset, 'seed': seed, 'split': 'test',
        'true_energy': np.asarray(data['Etrue'])[ti],
        'pred_energy': np.asarray(pred_energy),
        'pred_bias': np.nan if pred_bias is None else np.asarray(pred_bias),
        'region': reg, 'region_name': [REGION_NAME[int(r)] for r in reg],
        'ET': np.asarray(et)[ti]})
    if write:
        path = OUT / f'{dataset}__{model}.csv'
        df.to_csv(path, mode='a', header=not path.exists(), index=False)
    return df

### Model zoo (unified)
`Base`/`MeanDirect`/`MeanResidual`/`EfnResidual` (mean & EFN pooling, nb13) share an `nn.TransformerEncoder`. `PairT` (nb15/16/17) is the ParT-style pairwise-bias attention with optional `use_time_pair` (dt in U) and `use_gate` (in-time gated pooling). All residual models predict `base + head`; `MeanDirect` predicts directly.

In [9]:
CFG = dict(d=96, nhead=4, layers=3, dropout=0.1, lr=3e-4, wd=1e-4, batch=256, pair_hidden=32, huber_delta=0.1)
N_GLOBAL = 5
def encoder():
    layer = nn.TransformerEncoderLayer(CFG['d'], CFG['nhead'], dim_feedforward=4*CFG['d'],
                                       dropout=CFG['dropout'], batch_first=True)
    return nn.TransformerEncoder(layer, CFG['layers'], enable_nested_tensor=False)
def mlp_head(nf):
    return nn.Sequential(nn.Linear(nf, CFG['d']), nn.ReLU(), nn.Dropout(CFG['dropout']), nn.Linear(CFG['d'], 1))
class Base(nn.Module):
    def __init__(self, in_dim):
        super().__init__(); self.embed = nn.Linear(in_dim, CFG['d']); self.enc = encoder()
        self.norm = nn.LayerNorm(CFG['d']); self.head = mlp_head(CFG['d'] + N_GLOBAL)
    def encode(self, x, m): return self.enc(self.embed(x), src_key_padding_mask=~m)
class MeanDirect(Base):
    def forward(self, x, m, w, g, base):
        h = self.encode(x, m); wm = m.unsqueeze(-1).float()
        p = self.norm((h*wm).sum(1) / wm.sum(1).clamp(min=1)); return self.head(torch.cat([p, g], 1))
class MeanResidual(Base):
    def forward(self, x, m, w, g, base):
        h = self.encode(x, m); wm = m.unsqueeze(-1).float()
        p = self.norm((h*wm).sum(1) / wm.sum(1).clamp(min=1)); return base + self.head(torch.cat([p, g], 1))
class EfnResidual(Base):
    def forward(self, x, m, w, g, base):
        h = self.encode(x, m)
        return base + self.head(torch.cat([self.norm((h * w.unsqueeze(-1)).sum(1)), g], 1))

In [10]:
def pair_features(R, use_time):
    rx, ry, le = R[..., 0], R[..., 1], R[..., 2]
    dx = rx.unsqueeze(2) - rx.unsqueeze(1); dy = ry.unsqueeze(2) - ry.unsqueeze(1)
    dR = torch.sqrt(dx*dx + dy*dy + 1e-6)
    esum = le.unsqueeze(2) + le.unsqueeze(1); emin = torch.minimum(le.unsqueeze(2), le.unsqueeze(1))
    feats = [dx, dy, dR, esum, emin]
    if use_time:
        dt = R[..., 3]; hv = R[..., 4]
        feats += [(dt.unsqueeze(2) - dt.unsqueeze(1)).abs() * (hv.unsqueeze(2) * hv.unsqueeze(1)),
                  hv.unsqueeze(2) * hv.unsqueeze(1)]
    return torch.stack(feats, -1)
class PairEmbed(nn.Module):
    def __init__(self, n_in, nhead, hidden):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_in, hidden), nn.GELU(), nn.Linear(hidden, hidden), nn.GELU(),
                                 nn.Linear(hidden, nhead))
    def forward(self, pf): return self.net(pf).permute(0, 3, 1, 2).contiguous()
class PMHA(nn.Module):
    def __init__(self, d, nh, drop):
        super().__init__(); self.h = nh; self.dh = d // nh
        self.q = nn.Linear(d, d); self.k = nn.Linear(d, d); self.v = nn.Linear(d, d); self.o = nn.Linear(d, d)
        self.drop = nn.Dropout(drop)
    def forward(self, x, U, kv):
        B, L, d = x.shape
        q = self.q(x).view(B, L, self.h, self.dh).transpose(1, 2)
        k = self.k(x).view(B, L, self.h, self.dh).transpose(1, 2)
        v = self.v(x).view(B, L, self.h, self.dh).transpose(1, 2)
        s = (q @ k.transpose(-2, -1)) / (self.dh ** 0.5) + U
        s = s.masked_fill((~kv).view(B, 1, 1, L), -1e9)
        return self.o((self.drop(s.softmax(-1)) @ v).transpose(1, 2).reshape(B, L, d))
class Block(nn.Module):
    def __init__(self, d, nh, drop):
        super().__init__(); self.n1 = nn.LayerNorm(d); self.attn = PMHA(d, nh, drop); self.n2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 4*d), nn.GELU(), nn.Dropout(drop), nn.Linear(4*d, d)); self.drop = nn.Dropout(drop)
    def forward(self, x, U, kv):
        x = x + self.drop(self.attn(self.n1(x), U, kv)); return x + self.drop(self.ff(self.n2(x)))
class PairT(nn.Module):
    def __init__(self, in_dim, use_time_pair=False, use_gate=False):
        super().__init__(); d = CFG['d']; self.use_time_pair = use_time_pair; self.use_gate = use_gate
        self.embed = nn.Linear(in_dim, d); self.pair = PairEmbed(7 if use_time_pair else 5, CFG['nhead'], CFG['pair_hidden'])
        self.blocks = nn.ModuleList([Block(d, CFG['nhead'], CFG['dropout']) for _ in range(CFG['layers'])])
        self.norm = nn.LayerNorm(d); self.head = mlp_head(d + N_GLOBAL)
        if use_gate: self.gate = nn.Sequential(nn.Linear(2, 16), nn.GELU(), nn.Linear(16, 1))
    def forward(self, x, m, w, g, base, R):
        U = self.pair(pair_features(R, self.use_time_pair)); h = self.embed(x)
        for blk in self.blocks: h = blk(h, U, m)
        pw = w
        if self.use_gate:
            gate = torch.sigmoid(self.gate(R[..., 3:5]).squeeze(-1)) * m.float()
            pw = w * gate; pw = pw / pw.sum(1, keepdim=True).clamp(min=1e-9)
        p = self.norm((h * pw.unsqueeze(-1)).sum(1)); return base + self.head(torch.cat([p, g], 1))

### Data prep per dataset
Pads to tensors, z-scores the continuous columns on the train split, computes EFN pooling weights `w`, the 5-dim global features `g` and the calibrated-sum `base`. Split is fixed (seed 0); only model init varies with seed. `MODE=smoke` uses a few files for a GPU sanity run; `MODE=full` uses all files.

In [11]:
NFILES = {'smoke': 8, 'full': None}[MODE]
MODELS = {'smoke': ['EfnResidual', 'GateHuber'], 'full':
          ['MeanDirect','MeanResidual','EfnResidual','PairT','Spacetime','GateHuber']}[MODE]
USE_DATASETS = {'smoke': ['minbias'], 'full': ['clean','minbias']}[MODE]
SEEDS = {'smoke': [0], 'full': [0,1,2,3,4]}[MODE]
EPOCHS = {'smoke': 3, 'full': 150}[MODE]
PATIENCE = {'smoke': 99, 'full': 25}[MODE]
PAIR_KINDS = {'PairT': dict(dim='12'), 'Spacetime': dict(dim='15', use_time_pair=True),
              'GateHuber': dict(dim='15', use_gate=True)}
ENC_KINDS = {'MeanDirect': MeanDirect, 'MeanResidual': MeanResidual, 'EfnResidual': EfnResidual}
print('MODE', MODE, '| models', MODELS, '| datasets', USE_DATASETS, '| seeds', SEEDS, '| epochs', EPOCHS)

MODE full | models ['MeanDirect', 'MeanResidual', 'EfnResidual', 'PairT', 'Spacetime', 'GateHuber'] | datasets ['clean', 'minbias'] | seeds [0, 1, 2, 3, 4] | epochs 150


In [12]:
def prep(dataset):
    files = DATASETS[dataset] if NFILES is None else DATASETS[dataset][:NFILES]
    knn = lambda cc: select_knn(cc, 25)
    O = build_st(files, knn, 100.0)
    et = align_et(files, 100.0, knn)
    N = len(O['Etrue']); assert len(et) == N, ('MISALIGNED', len(et), N)
    Et = O['Etrue']; y = O['y'].astype(np.float32); agg = O['agg']; region = O['region']
    keep = np.flatnonzero((Et >= 1.0) & (Et <= 100.0))
    ktr, kva, kte = (keep[s] for s in split(len(keep)))
    G = np.stack([agg[:,0], agg[:,3], np.log(agg[:,2]+1.0), agg[:,1], agg[:,4]], 1).astype(np.float32)
    G = (G - G[ktr].mean(0)) / (G[ktr].std(0) + EPS)
    la, lb = np.polyfit(agg[ktr,0], y[ktr], 1); base_all = (la*agg[:,0] + lb).astype(np.float32)
    maxL = max(t.shape[0] for t in O['tok15'])
    def pad(toks, F):
        A = np.zeros((N, maxL, F), np.float32)
        for i, t in enumerate(toks): A[i, :t.shape[0]] = t
        return A
    X12 = pad(O['tok12'], 12); X15 = pad(O['tok15'], 15); Rp = pad(O['R'], 5)
    M = np.zeros((N, maxL), np.bool_); W = np.zeros((N, maxL), np.float32)
    for i, t in enumerate(O['tok12']):
        L = t.shape[0]; M[i, :L] = True
        e = np.expm1(np.clip(t[:,0], 0, None)); W[i, :L] = e / (e.sum() + 1e-9)
    def stdz(A, ncont):
        cont = A[ktr][:, :, :ncont].reshape(-1, ncont)[M[ktr].reshape(-1)]
        mean = cont.mean(0); std = cont.std(0) + EPS
        A = A.copy(); A[:, :, :ncont] = (A[:, :, :ncont] - mean) / std; A[~M] = 0.0; return A
    X12 = stdz(X12, 7); X15 = stdz(X15, 9)
    t = lambda z: torch.from_numpy(z).to(DEVICE)
    P = dict(X12=t(X12), X15=t(X15), R=t(Rp), M=t(M), W=t(W), G=t(G),
             base=t(base_all).unsqueeze(1), y=t(y).unsqueeze(1),
             ktr=ktr, kva=kva, kte=kte, Et=Et, base_all=base_all, region=region, et=et, O=O,
             la=la, lb=lb, agg=agg)
    print(f'{dataset:8s} N={N} kept(1-100GeV)={len(keep)} maxL={maxL} '
          f'train/val/test={len(ktr)}/{len(kva)}/{len(kte)}')
    return P

### Unified train / eval
MSE on `y=log(Etrue)` (Huber for GateHuber), AdamW + cosine, early-stop on val loss. Calibrate raw output on val (`a,b`), then `pred_energy=exp(a*raw+b)`; `pred_bias` = the learned log-residual `raw-base` for residual models, NaN for `MeanDirect`.

In [13]:
def train_eval(kind, P, seed):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    is_pair = kind in PAIR_KINDS
    if is_pair:
        spec = PAIR_KINDS[kind]; Xt = P['X15'] if spec['dim'] == '15' else P['X12']
        model = PairT(int(spec['dim']), spec.get('use_time_pair', False), spec.get('use_gate', False)).to(DEVICE)
    else:
        Xt = P['X12']; model = ENC_KINDS[kind](12).to(DEVICE)
    residual = kind != 'MeanDirect'
    huber = kind == 'GateHuber'
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['wd'])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    def fwd(b):
        args = (Xt[b], P['M'][b], P['W'][b], P['G'][b], P['base'][b])
        return model(*args, P['R'][b]) if is_pair else model(*args)
    def lossf(p, t):
        return nn.functional.huber_loss(p, t, delta=CFG['huber_delta']) if huber else nn.functional.mse_loss(p, t)
    def batches(idx, bs, shuffle):
        idx = np.asarray(idx)
        if shuffle: idx = rng.permutation(idx)
        for j in range(0, len(idx), bs):
            yield torch.from_numpy(idx[j:j+bs]).to(DEVICE)
    def run(idx):
        out = []
        with torch.no_grad():
            for b in batches(idx, 512, False): out.append(fwd(b).cpu().numpy().ravel())
        return np.concatenate(out)
    def vloss():
        model.eval(); s = 0.0; n = 0
        with torch.no_grad():
            for b in batches(P['kva'], 512, False): s += lossf(fwd(b), P['y'][b]).item(); n += 1
        return s / max(n, 1)
    best = 1e9; bstate = None; wait = 0
    for ep in range(EPOCHS):
        model.train()
        for b in batches(P['ktr'], CFG['batch'], True):
            opt.zero_grad(); lossf(fwd(b), P['y'][b]).backward(); opt.step()
        sched.step(); vv = vloss()
        if vv < best - 1e-4: best = vv; bstate = copy.deepcopy(model.state_dict()); wait = 0
        else:
            wait += 1
            if wait >= PATIENCE: break
    model.load_state_dict(bstate); model.eval()
    a, b = np.polyfit(run(P['kva']), P['y'].cpu().numpy().ravel()[P['kva']], 1)
    raw_te = run(P['kte']); pe = np.exp(a * raw_te + b)
    bias = (raw_te - P['base_all'][P['kte']]) if residual else None
    sig = float(resolution(pe, P['Et'][P['kte']])['sigma_eff'])
    return sig, pe, bias

### Baselines (BDT, CalibratedSum)

In [14]:
def run_baselines(P, dataset, write):
    kte = P['kte']; Et = P['Et']; agg = P['agg']
    pe_sum = np.exp(P['base_all'][kte])
    save_predictions(pe_sum, None, P['O'], P['et'], 'CalibratedSum', dataset, 0, kte, write=write)
    ssum = float(resolution(pe_sum, Et[kte])['sigma_eff'])
    gb = HistGradientBoostingRegressor(max_iter=300, random_state=0).fit(agg[P['ktr']], P['y'].cpu().numpy().ravel()[P['ktr']])
    pe_bdt = np.exp(gb.predict(agg[kte]))
    save_predictions(pe_bdt, None, P['O'], P['et'], 'BDT', dataset, 0, kte, write=write)
    sbdt = float(resolution(pe_bdt, Et[kte])['sigma_eff'])
    print(f'  [{dataset}] CalibratedSum {ssum:.4f} | BDT {sbdt:.4f}')

### Run: all models x datasets x seeds
For a clean re-run, any existing CSV for a `(dataset, model)` is deleted before seed 0 so appends do not double up. Set `NB19_MODE=smoke` for a quick GPU sanity check; default `full` runs everything.

In [15]:
import time
def csv_complete(dataset, model):
    p = OUT / f'{dataset}__{model}.csv'
    if not p.exists(): return False
    if model in ('BDT', 'CalibratedSum'): return True
    try: return pd.read_csv(p, usecols=['seed'])['seed'].nunique() >= len(SEEDS)
    except Exception: return False
for dataset in USE_DATASETS:
    resume = MODE == 'full'
    base_need = resume and not (csv_complete(dataset, 'BDT') and csv_complete(dataset, 'CalibratedSum'))
    need = [m for m in MODELS if not (resume and csv_complete(dataset, m))]
    if resume and not need and not base_need:
        print(f'=== skip {dataset} (all complete) ===', flush=True); continue
    print(f'=== building {dataset} | need {need} | baselines {base_need or not resume} ===', flush=True)
    P = prep(dataset)
    if base_need or not resume:
        run_baselines(P, dataset, write=(MODE == 'full'))
    for kind in need:
        if MODE == 'full':
            (OUT / f'{dataset}__{kind}.csv').unlink(missing_ok=True)
        sigs = []; t0 = time.time()
        for seed in SEEDS:
            sig, pe, bias = train_eval(kind, P, seed)
            sigs.append(sig)
            save_predictions(pe, bias, P['O'], P['et'], kind, dataset, seed, P['kte'], write=(MODE == 'full'))
        print(f'  [{dataset}] {kind:14s} sigma_eff {np.mean(sigs):.4f} +/- {np.std(sigs):.4f}  '
              f'({len(SEEDS)} seeds, {time.time()-t0:.0f}s)', flush=True)
print('DONE mode=' + MODE)

=== skip clean (all complete) ===


=== building minbias | need ['EfnResidual', 'PairT', 'Spacetime', 'GateHuber'] | baselines False ===


minbias  N=89797 kept(1-100GeV)=81509 maxL=25 train/val/test=57056/12226/12227


  [minbias] EfnResidual    sigma_eff 0.0660 +/- 0.0024  (5 seeds, 2122s)


  [minbias] PairT          sigma_eff 0.0663 +/- 0.0013  (5 seeds, 3784s)


  [minbias] Spacetime      sigma_eff 0.0575 +/- 0.0017  (5 seeds, 2930s)


  [minbias] GateHuber      sigma_eff 0.0485 +/- 0.0008  (5 seeds, 4043s)


DONE mode=full


In [16]:
summary = []
for p in sorted(OUT.glob('*.csv')):
    df = pd.read_csv(p)
    for (ds, mdl), g in df.groupby(['dataset', 'model']):
        vals = [resolution(gg['pred_energy'].to_numpy(), gg['true_energy'].to_numpy())['sigma_eff']
                for _, gg in g.groupby('seed')]
        summary.append((ds, mdl, round(np.mean(vals), 4), round(np.std(vals), 4), g['seed'].nunique()))
S = pd.DataFrame(summary, columns=['dataset', 'model', 'sigma_eff', 'std', 'seeds']).sort_values(['dataset', 'sigma_eff'])
print(S.to_string(index=False))

dataset         model  sigma_eff    std  seeds
  clean  MeanResidual     0.0403 0.0017      5
  clean   EfnResidual     0.0407 0.0012      5
  clean         PairT     0.0413 0.0014      5
  clean     Spacetime     0.0423 0.0006      5
  clean     GateHuber     0.0448 0.0011      5
  clean    MeanDirect     0.0505 0.0021      5
  clean           BDT     0.0528 0.0000      1
  clean CalibratedSum     0.0822 0.0000      1
minbias     GateHuber     0.0485 0.0008      5
minbias     Spacetime     0.0575 0.0017      5
minbias   EfnResidual     0.0660 0.0024      5
minbias         PairT     0.0663 0.0013      5
minbias    MeanDirect     0.0696 0.0017      5
minbias  MeanResidual     0.0713 0.0013      5
minbias           BDT     0.1253 0.0000      1
minbias CalibratedSum     0.1837 0.0000      1


**Outputs:** `reports/predictions/{dataset}__{model}.csv` for every architecture and both datasets, with columns `model, dataset, seed, split, true_energy, pred_energy, pred_bias, region, region_name, ET`. nb20 loads these for the comparison plots.